In [4]:
import os
import zipfile
import shutil
import io
import pandas as pd
pd.set_option('display.max_columns', None)

### Mimic the POI data point extraction to find out how this was done for: https://arxiv.org/pdf/2506.00765

In [ ]:
# from ohsome import OhsomeClient
# import pandas as pd, numpy as np, time, os, math, json
# from geopy.geocoders import Nominatim
# from geopy.extra.rate_limiter import RateLimiter
# from concurrent.futures import ThreadPoolExecutor, as_completed
# from datetime import datetime
# from typing import Dict, Tuple, List

# # ---------- Config ----------
# ZIP_CODES = ["10002", "70302"]  # replace with your list
# YEAR_FROM, YEAR_TO = 2017, 2018
# BASE_URL = "https://api.ohsome.org/v1"   # or your local ohsome
# MAX_WORKERS = 8                          # be polite; 2–4 is usually safe
# SLEEP_BETWEEN_CALLS = 0.25               # seconds between API calls
# OUT_DIR = "poi_out"                      # per-ZIP outputs here

# POI_FILTERS = {
#     "bank":        "amenity=bank",
#     "bus":         "highway=bus_stop OR (public_transport=platform AND bus=yes)",
#     "hospital":    "amenity=hospital OR amenity=clinic",
#     "mall":        "shop=mall OR landuse=retail",
#     "park":        "leisure=park OR boundary=national_park OR landuse=grass",
#     "restaurant":  "amenity=restaurant OR amenity=fast_food OR amenity=cafe",
#     "school":      "amenity=school OR amenity=college OR amenity=university",
#     "station":     "railway=station OR public_transport=station",
#     "supermarket": "shop=supermarket OR amenity=marketplace",
# }

# os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# def already_done(zip_code: str) -> bool:
#     return os.path.exists(os.path.join(OUT_DIR, f"{zip_code}_poi_counts_monthly.csv"))

# def process_zip(zip_code: str) -> Tuple[str, str]:
#     """
#     Worker: fetch, save per-ZIP CSV (raw). Returns (zip, status).
#     Per-ZIP write lets you resume without redoing finished ZIPs.
#     """
#     try:
#         if already_done(zip_code):
#             return (zip_code, "skipped (exists)")
#         df = collect_zip_monthly_pois(zip_code)
#         # Save per-ZIP raw (month index → column)
#         out = df.reset_index(names="month")
#         out.to_csv(os.path.join(OUT_DIR, f"{zip_code}_poi_counts_monthly.csv"), index=False)
#         return (zip_code, "ok")
#     except Exception as e:
#         return (zip_code, f"error: {e}")

# def build_poi_panel_parallel(zip_codes=ZIP_CODES, max_workers=MAX_WORKERS) -> Tuple[pd.DataFrame, pd.DataFrame]:
#     """
#     Runs ZIPs in parallel, then merges all per-ZIP CSVs, applies global imputation + log1p,
#     and writes panel CSVs.
#     """
#     # 1) Parallel per-ZIP fetch → impute-within → save
#     todo = [z for z in zip_codes if not already_done(z)]
#     if todo:
#         with ThreadPoolExecutor(max_workers=max_workers) as ex:
#             futures = {ex.submit(process_zip, z): z for z in todo}
#             for f in as_completed(futures):
#                 z = futures[f]
#                 status = f.result()
#                 print(status)

#     # 2) Merge all per-ZIP CSVs
#     frames = []
#     for z in zip_codes:
#         p = os.path.join(OUT_DIR, f"{z}_poi_counts_monthly.csv")
#         if not os.path.exists(p):
#             raise FileNotFoundError(f"Missing expected per-ZIP CSV for {z}: {p}")
#         frames.append(pd.read_csv(p, parse_dates=["month"]))
#     panel = pd.concat(frames, ignore_index=True).sort_values(["zip", "month"])

#     # 3) Global imputation + log
#     panel_log = log_transform(panel)

#     # 4) Save combined panels
#     panel.to_csv(os.path.join(OUT_DIR, "poi_counts_monthly_panel.csv"), index=False)
#     panel_log.to_csv(os.path.join(OUT_DIR, "poi_counts_monthly_panel_log.csv"), index=False)
#     return panel, panel_log


# # -------------------------------
# # UTILITIES
# # -------------------------------

# def geocode_zip_bbox(zip_code: str, country="USA", pause=1.0) -> List[float]:
#     """Nominatim → bbox as [minLon, minLat, maxLon, maxLat]. Cache results to avoid re-geocoding."""
#     cache_path = os.path.join(OUT_DIR, "bbox_cache.json")
#     cache = {}
#     if os.path.exists(cache_path):
#         try:
#             cache = json.load(open(cache_path, "r"))
#         except Exception:
#             cache = {}
#     if zip_code in cache:
#         return cache[zip_code]

#     geolocator = Nominatim(user_agent="poi-pipeline-ohsomepy", timeout=10)
#     rate = RateLimiter(geolocator.geocode, min_delay_seconds=pause, max_retries=3, error_wait_seconds=2)
#     loc = rate(f"{zip_code}, {country}", addressdetails=True, exactly_one=True)
#     if not loc or "boundingbox" not in getattr(loc, "raw", {}):
#         raise ValueError(f"Could not geocode ZIP: {zip_code}")
#     south, north, west, east = map(float, loc.raw["boundingbox"])
#     bbox = [west, south, east, north]
#     cache[zip_code] = bbox
#     with open(cache_path, "w") as f:
#         json.dump(cache, f)
#     return bbox

# def year_interval(y: int) -> str:
#     return f"{y}-01-01/{y}-12-31/P1M"  # monthly buckets

# def backoff_sleep(attempt: int):
#     # 0, 1, 2, 3, 4 → 0.5s, 1s, 2s, 4s, 8s (capped)
#     time.sleep(min(0.5 * (2 ** attempt), 8.0))

# def fetch_series_for_filter(client: OhsomeClient, bbox: List[float], filter_expr: str,
#                             year_from=YEAR_FROM, year_to=YEAR_TO,
#                             pause=SLEEP_BETWEEN_CALLS,
#                             max_retries=5) -> pd.Series:
#     """Hits /elements/count/series year-by-year; returns a monthly Series."""
#     pieces = []
#     for y in range(year_from, year_to + 1):
#         time_param = year_interval(y)
#         attempt = 0
#         while True:
#             try:
#                 resp = client.post(
#                     endpoint="elements/count/series",
#                     bboxes=bbox,      # [minLon, minLat, maxLon, maxLat]
#                     time=time_param,
#                     filter=filter_expr,
#                 )
#                 df = resp.as_dataframe()  # columns: timestamp, value
#                 s = pd.Series(
#                     df["value"].to_numpy(),
#                     index=pd.to_datetime(df["timestamp"]).tz_localize(None),
#                     dtype="float",
#                 ).asfreq("MS")
#                 pieces.append(s)
#                 time.sleep(pause)
#                 break
#             except Exception as e:
#                 if attempt >= max_retries:
#                     raise RuntimeError(f"Failed fetching {filter_expr} for {y}: {e}") from e
#                 backoff_sleep(attempt)
#                 attempt += 1
#                 continue
#     if not pieces:
#         return pd.Series(dtype="float")
#     out = pd.concat(pieces).sort_index()
#     return out[~out.index.duplicated(keep="first")]

# def collect_zip_monthly_pois(zip_code: str) -> pd.DataFrame:
#     """Collect all categories for one ZIP (with its own OhsomeClient)."""
#     bbox = geocode_zip_bbox(zip_code)
#     client = OhsomeClient(base_api_url=BASE_URL)  # new client per worker
#     cols = []
#     for cat, filt in POI_FILTERS.items():
#         s = fetch_series_for_filter(client, bbox, filt)
#         s.name = cat
#         cols.append(s)
#     df = pd.concat(cols, axis=1).sort_index()
#     df.insert(0, "zip", zip_code)
#     return df

# def log_transform(panel: pd.DataFrame) -> pd.DataFrame:
#     num_cols = [c for c in panel.columns if c not in ("month", "zip")]
#     out = panel.copy()
#     out[num_cols] = np.log1p(out[num_cols].astype(float))
#     return out

## Gather Zillow Data

In [2]:
!pwd

/mnt/c/Users/karanveer/Desktop/MIDSCapstone/RentIQ/EDA


In [62]:
# Pull in Zillow data
zillow_df = pd.read_csv("/mnt/c/Users/karanveer/Desktop/MIDSCapstone/RentIQ/EDA/Datasets/Zillow/zillow_rental_data.csv")

In [63]:
# Zillow data for NYC
zillow_ny = zillow_df[zillow_df['State']=='NY']

In [64]:
# Select relvevant columns
zillow_cols = ['RegionName', 'City', 'CountyName'] + list(zillow_ny.columns[33:117])
zillow_ny = zillow_ny[zillow_cols]

# Rename the zip column
zillow_ny = zillow_ny.rename(columns={"RegionName" : "zipcode"})

# Keep only non-null values from 2017-2023 (consistent with the range of HouseTS data)
zillow_ny = zillow_ny[zillow_ny['2017-01-31'].notna()].reset_index(drop=True)

In [66]:
# Convert wide data to long data
zillow_long = zillow_ny.melt(
    id_vars=["zipcode", "City", "CountyName"],      # Keep ZipCode fixed
    var_name="date",          # Column name for former date-columns
    value_name="median_rent"         # Column name for values
)

# Convert Date column to datetime type
zillow_long["date"] = pd.to_datetime(zillow_long["date"])

In [67]:
zillow_long = zillow_long.rename(columns={'City' : 'city',
                                          'CountyName' : 'county'})

In [68]:
# Create a list of NYC zip codes
zillow_zips = zillow_long['zipcode'].unique().tolist()

## Gather HouseTS Dataset

In [69]:
# Pull data HouseTS Dataset
house_data = pd.read_csv("/mnt/c/Users/karanveer/Desktop/MIDSCapstone/RentIQ/EDA/Datasets/HouseTS/HouseTS.csv")

In [70]:
# Look at the shape
house_data.shape

(884092, 39)

In [71]:
# What features are available
house_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 884092 entries, 0 to 884091
Data columns (total 39 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   date                          884092 non-null  object 
 1   median_sale_price             884092 non-null  float64
 2   median_list_price             884092 non-null  float64
 3   median_ppsf                   884092 non-null  float64
 4   median_list_ppsf              884092 non-null  float64
 5   homes_sold                    884092 non-null  float64
 6   pending_sales                 884092 non-null  float64
 7   new_listings                  884092 non-null  float64
 8   inventory                     884092 non-null  float64
 9   median_dom                    884092 non-null  float64
 10  avg_sale_to_list              884092 non-null  float64
 11  sold_above_list               884092 non-null  float64
 12  off_market_in_two_weeks       884092 non-nul

In [72]:
# View dataframe head
house_data.head()

,date,median_sale_price,median_list_price,median_ppsf,median_list_ppsf,homes_sold,pending_sales,new_listings,inventory,median_dom,avg_sale_to_list,sold_above_list,off_market_in_two_weeks,city,zipcode,year,bank,bus,hospital,mall,park,restaurant,school,station,supermarket,Total Population,Median Age,Per Capita Income,Total Families Below Poverty,Total Housing Units,Median Rent,Median Home Value,Total Labor Force,Unemployed Population,Total School Age Population,Total School Enrollment,Median Commute Time,price,city_full
0,2012-03-31,46550.0,217450.0,31.813674,110.183666,14.0,23.0,44.0,64.0,59.5,0.943662,0.142857,0.043478,ATL,30002,2012,12.0,2.0,4.0,1.0,60.0,45.0,57.0,4.0,7.0,5811.0,36.3,33052.0,5811.0,2677.0,710.0,279500.0,3171.0,460.0,5408.0,5408.0,2492.0,200773.999557,Atlanta-Sandy Springs-Alpharetta
1,2012-04-30,61870.0,245000.0,40.723982,130.528256,22.0,29.0,56.0,69.0,89.5,0.946642,0.090909,0.034483,ATL,30002,2012,12.0,2.0,4.0,1.0,60.0,45.0,57.0,4.0,7.0,5811.0,36.3,33052.0,5811.0,2677.0,710.0,279500.0,3171.0,460.0,5408.0,5408.0,2492.0,202421.064584,Atlanta-Sandy Springs-Alpharetta
2,2012-05-31,125500.0,217450.0,63.913043,119.919216,24.0,40.0,63.0,60.0,144.5,0.955624,0.208333,0.100000,ATL,30002,2012,12.0,2.0,4.0,1.0,60.0,45.0,57.0,4.0,7.0,5811.0,36.3,33052.0,5811.0,2677.0,710.0,279500.0,3171.0,460.0,5408.0,5408.0,2492.0,202681.309539,Atlanta-Sandy Springs-Alpharetta
3,2012-06-30,153000.0,189900.0,81.598080,105.617353,34.0,46.0,50.0,57.0,126.0,0.970608,0.176471,0.108696,ATL,30002,2012,12.0,2.0,4.0,1.0,60.0,45.0,57.0,4.0,7.0,5811.0,36.3,33052.0,5811.0,2677.0,710.0,279500.0,3171.0,460.0,5408.0,5408.0,2492.0,202998.603897,Atlanta-Sandy Springs-Alpharetta
4,2012-07-31,165500.0,154000.0,81.598080,83.921175,39.0,49.0,42.0,50.0,80.0,0.982105,0.256410,0.102041,ATL,30002,2012,12.0,2.0,3.0,1.0,60.0,45.0,57.0,4.0,6.0,5811.0,36.3,33052.0,5811.0,2677.0,710.0,279500.0,3171.0,460.0,5408.0,5408.0,2492.0,203781.903446,Atlanta-Sandy Springs-Alpharetta


In [73]:
# Convert the date column to datetime object
house_data['date'] = pd.to_datetime(house_data['date'])

In [74]:
# Filter the TS data to records that exist in the Zillow dataset
house_data_nyc = house_data[house_data['city']=='NY']

In [75]:
# Filter the data to start from 2017
house_data_nyc = house_data_nyc[house_data_nyc['date'] >= pd.to_datetime('2017-01-01')].reset_index(drop=True)

In [76]:
# Retain only the zipcodes that are in the zillow rentals dataset
house_data_nyc = house_data_nyc[house_data_nyc['zipcode'].isin(zillow_zips)]

In [81]:
house_data_nyc.shape

(6300, 39)

In [83]:
house_data_nyc.shape

(529200, 43)

In [80]:
zillow_long

,zipcode,city,county,date,median_rent
0,11385,New York,Queens County,2017-01-31,2303.742913
1,10467,New York,Bronx County,2017-01-31,1320.387659
2,11226,New York,Kings County,2017-01-31,2333.401703
3,11207,New York,Kings County,2017-01-31,2184.043420
4,10025,New York,New York County,2017-01-31,3184.747277
...,...,...,...,...,...
6547,10007,New York,New York County,2023-12-31,7362.615226
6548,10018,New York,New York County,2023-12-31,4392.677529
6549,10280,New York,New York County,2023-12-31,4535.575909
6550,11109,New York,Queens County,2023-12-31,4201.434278


In [82]:
house_data_nyc = house_data_nyc.merge(
    right=zillow_long,
    how='left',
    on='zipcode',
    suffixes=("_ts", "_zlw")
)